# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imaniftikhar/week1_flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [3]:
import os, getpass
import duckdb
import pandas as pd
import numpy as np

# 1. Authenticate & Connect DuckDB to Hugging Face Warehouse
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN.strip()}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact_table = f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')"
dim_content = f"read_parquet('{REL}/dim_content.parquet')"

# 2. Dynamic Feature & Target Windowing (Lane 2: Opportunity Scoring)
df_raw = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS max_d FROM {fact_table})
    SELECT
        f.content_hash_id,
        f.client_hash_id,

        -- Baseline Historical Window (Prior performance)
        SUM(CASE WHEN f.report_date <= b.max_d - INTERVAL 15 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev,
        SUM(CASE WHEN f.report_date <= b.max_d - INTERVAL 15 DAY THEN f.gsc_clicks ELSE 0 END) AS clk_prev,
        AVG(CASE WHEN f.report_date <= b.max_d - INTERVAL 15 DAY AND f.gsc_avg_position > 0 THEN f.gsc_avg_position END) AS pos_prev,

        -- Target Evaluation Window (Recent performance)
        SUM(CASE WHEN f.report_date > b.max_d - INTERVAL 15 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_recent,

        -- Content Attributes
        ANY_VALUE(c.word_count) AS word_count,
        ANY_VALUE(c.content_type) AS content_type
    FROM {fact_table} f
    CROSS JOIN bounds b
    LEFT JOIN {dim_content} c ON f.content_hash_id = c.content_hash_id
    GROUP BY f.content_hash_id, f.client_hash_id
    HAVING SUM(CASE WHEN f.report_date <= b.max_d - INTERVAL 15 DAY THEN f.gsc_impressions ELSE 0 END) >= 10
""").df()

# 3. Target Label (Needs Refresh if recent impressions dropped by >20% relative to baseline)
df_raw['needs_refresh'] = (df_raw['imp_recent'] < 0.8 * df_raw['imp_prev']).astype(int)

# 4. Construct Feature Vector (Pre-decision signals only)
features = pd.DataFrame()

# Feature 1: Baseline Impression Volume
features['imp_prev'] = df_raw['imp_prev']

# Feature 2: Baseline Click-Through Rate
features['ctr_prev'] = (df_raw['clk_prev'] / df_raw['imp_prev'].replace(0, np.nan)).fillna(0)

# Feature 3: Baseline Average Search Rank (Fill missing/zero rank with default 20.0)
features['pos_prev'] = df_raw['pos_prev'].fillna(20.0)

# Feature 4: Word Count (Fill missing with median)
features['word_count'] = df_raw['word_count'].fillna(df_raw['word_count'].median())

# Feature 5: Content Type One-Hot Encoding
type_dummies = pd.get_dummies(df_raw['content_type'], prefix='type', drop_first=True)
features = pd.concat([features, type_dummies], axis=1)

y = df_raw['needs_refresh']
groups = df_raw['client_hash_id']

print(f"Feature vector constructed: {features.shape[0]:,} rows across {features.shape[1]} features.")
features.head()

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector constructed: 139,347 rows across 6 features.


,imp_prev,ctr_prev,pos_prev,word_count,type_feedly article,type_keyword article
0,118.0,0.000000,45.745611,963,False,True
1,51.0,0.019608,15.901282,1109,False,True
2,16.0,0.000000,18.321429,1070,False,True
3,139.0,0.007194,44.940274,1137,False,True
4,65.0,0.000000,40.574913,990,False,True


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

- imp_prev: Total Google Search Console impressions logged during the historical baseline window. Missing values are filtered out at query time (HAVING imp_prev >= 10). Available before decision: Yes, collected prior to the prediction timestamp.

- ctr_prev: Baseline Click-Through Rate calculated as historical clicks divided by historical impressions (clk_prev / imp_prev). Division by zero is safely filled with 0. Available before decision: Yes, computed strictly from past activity.

- pos_prev: Historical average organic search rank position (excluding non-impression days where rank is recorded as 0). Missing values or non-ranked URLs are imputed with a default rank of 20.0. Available before decision: Yes, logged continuously prior to evaluation.

- word_count: Total word count extracted from published article metadata in dim_content. Missing values are imputed using the overall dataset median word count. Available before decision: Yes, extracted directly from existing HTML prior to triage.

- type_*: One-hot encoded categorical indicators representing the content archetype (e.g., blog, landing page, guide). Missing or unmatched categories default to 0. Available before decision: Yes, static taxonomy attribute stored in CMS metadata.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupShuffleSplit

# Split data using GroupShuffleSplit on client_hash_id to avoid cross-client leakage
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(features, y, groups))

X_tr, X_te = features.iloc[train_idx].copy(), features.iloc[test_idx].copy()
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

# --- TEST 1: Honest Baseline Model ---
honest_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
honest_model.fit(X_tr, y_tr)
honest_acc = accuracy_score(y_te, honest_model.predict(X_te))

print(f"Honest Model Accuracy (Unseen Clients): {honest_acc:.4f}")

# --- TEST 2: Inject Leaked Feature (Post-Decision Target Window Signal) ---
X_tr_leaked = X_tr.copy()
X_te_leaked = X_te.copy()

X_tr_leaked['leaked_imp_recent'] = df_raw.iloc[train_idx]['imp_recent']
X_te_leaked['leaked_imp_recent'] = df_raw.iloc[test_idx]['imp_recent']

leaked_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
leaked_model.fit(X_tr_leaked, y_tr)
leaked_acc = accuracy_score(y_te, leaked_model.predict(X_te_leaked))

print(f"Leaked Model Accuracy (With Future Signal): {leaked_acc:.4f} (Artificially Inflated!)")
print("-" * 65)
print(f"Leakage Impact: Accuracy jumped from {honest_acc:.4f} -> {leaked_acc:.4f}")



Honest Model Accuracy (Unseen Clients): 0.5216
Leaked Model Accuracy (With Future Signal): 0.9732 (Artificially Inflated!)
-----------------------------------------------------------------
Leakage Impact: Accuracy jumped from 0.5216 -> 0.9732


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- imp_recent / imp_last30: Excluded. Represents metrics from the post-decision target evaluation window used directly in the math to generate the needs_refresh label. Including it causes catastrophic target leakage (jumping accuracy artificially from 0.5216 to 0.9732).

- trend_direction & trend_pct: Excluded. Pre-calculated performance trends derived post-hoc across the prediction boundary, which directly leak whether content is decaying in the evaluation window.

- health_score & action_taken: Excluded. Downstream system intervention flags recorded after triage evaluation, creating circular, non-causal target leakage.

- content_hash_id & client_hash_id: Excluded. Arbitrary primary keys that lead to severe client-level overfitting rather than learning generalizable signals across unseen sites.

- ga4_* metrics (unfiltered): Excluded. Historical rows prior to GA4 deployment contain zero-filled placeholders, which introduce measurement bias unless explicitly isolated by filtering on ga4_data_available IS TRUE.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.